In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
import os

print("--- Searching /kaggle/input directory ---")
for root, dirs, files in os.walk("/kaggle/input"):
    for file in files:
        if file.endswith(('.csv', '.tsv')):
            print(f"CSV Found: {os.path.join(root, file)}")

In [1]:
# ============================================================
# FINAL MODEL for IMUSA / HASOC Task-2 (Punjabi Memes)
# Focus: Best Accuracy + Precision + Recall + F1
# ============================================================

import os, copy, warnings, gc
import numpy as np
import pandas as pd
from PIL import Image
from collections import Counter

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from transformers import AutoTokenizer, AutoModel, logging as tf_logging
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report, accuracy_score, f1_score

warnings.filterwarnings("ignore")
tf_logging.set_verbosity_error()
torch.cuda.empty_cache()
gc.collect()

class Config:
    TEXT_MODEL  = "l3cube-pune/punjabi-bert"
    MAX_LEN     = 128
    BATCH_SIZE  = 16
    EPOCHS      = 12
    LR          = 2e-5
    DEVICE      = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    OUTPUT_DIR  = "/kaggle/working"
    SEED        = 42

    TRAIN_CSV = "/kaggle/input/datasets/yatharv/hesoc-data/Task2/Training_Dataset/Training_Dataset/train_punjabi_dataset.csv"
    TRAIN_IMG = "/kaggle/input/datasets/yatharv/hesoc-data/Task2/Training_Dataset/Training_Dataset/Training_images"
    TEST_CSV  = "/kaggle/input/datasets/yatharv/hesoc-data/Task2/Testing_Dataset/Testing_Dataset/Test.csv"
    TEST_IMG  = "/kaggle/input/datasets/yatharv/hesoc-data/Task2/Testing_Dataset/Testing_Dataset/Testing_images"

    LABEL_MAP = {
        "sarcasm": 0, "sarcastic": 0,
        "neutral": 1,
        "offensive": 2, "offense": 2,
        "motivational": 3, "motivation": 3
    }
    REV_LABEL = {0: "Sarcasm", 1: "Neutral", 2: "Offensive", 3: "Motivational"}

torch.manual_seed(Config.SEED)
np.random.seed(Config.SEED)

# -------------------- Transforms --------------------
train_tf = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.RandomCrop(224),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(0.2, 0.2, 0.2, 0.05),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

val_tf = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

# -------------------- Dataset --------------------
class MemeDataset(Dataset):
    def __init__(self, df, img_dir, tokenizer, transform, is_test=False):
        self.df = df.reset_index(drop=True)
        self.img_dir = img_dir
        self.tokenizer = tokenizer
        self.transform = transform
        self.is_test = is_test

        cols = {str(c).lower().strip(): c for c in df.columns}
        self.text_col = next((cols[c] for c in ['text', 'post', 'caption', 'transcription'] if c in cols), df.columns[1])
        self.img_col  = next((cols[c] for c in ['image_name', 'image_id', 'image', 'filename'] if c in cols), df.columns[0])
        self.id_col   = next((cols[c] for c in ['id', 'image_id', 's_no'] if c in cols), self.img_col)
        if not is_test:
            self.label_col = next((cols[c] for c in ['category', 'label', 'class'] if c in cols), df.columns[-1])

    def __len__(self):
        return len(self.df)

    def _load_image(self, name):
        path = os.path.join(self.img_dir, str(name))
        if not os.path.exists(path):
            for ext in ['.jpg', '.jpeg', '.png', '.JPG', '.JPEG', '.PNG']:
                if os.path.exists(path + ext):
                    path += ext
                    break
        try:
            img = Image.open(path).convert('RGB')
        except:
            img = Image.new('RGB', (224, 224), (0, 0, 0))
        return self.transform(img)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        text = str(row[self.text_col]) if pd.notna(row[self.text_col]) else ""
        enc = self.tokenizer(text, truncation=True, padding='max_length',
                             max_length=Config.MAX_LEN, return_tensors='pt')

        item = {
            'input_ids': enc['input_ids'].squeeze(0),
            'attention_mask': enc['attention_mask'].squeeze(0),
            'image': self._load_image(row[self.img_col]),
            'id': str(row[self.id_col])
        }
        if not self.is_test:
            lab = str(row[self.label_col]).strip().lower()
            item['label'] = torch.tensor(Config.LABEL_MAP.get(lab, 1), dtype=torch.long)
        return item

# -------------------- Model --------------------
class MultimodalClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.text_encoder = AutoModel.from_pretrained(Config.TEXT_MODEL)
        self.image_encoder = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)
        self.image_encoder.fc = nn.Identity()

        self.classifier = nn.Sequential(
            nn.Linear(768 + 2048, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(512, 4)
        )

    def forward(self, input_ids, attention_mask, image):
        t = self.text_encoder(input_ids=input_ids, attention_mask=attention_mask).last_hidden_state[:, 0]
        v = self.image_encoder(image)
        return self.classifier(torch.cat([t, v], dim=1))

# -------------------- Train / Eval --------------------
def train_epoch(model, loader, criterion, optimizer):
    model.train()
    total = 0
    for batch in loader:
        optimizer.zero_grad()
        logits = model(
            batch['input_ids'].to(Config.DEVICE),
            batch['attention_mask'].to(Config.DEVICE),
            batch['image'].to(Config.DEVICE)
        )
        loss = criterion(logits, batch['label'].to(Config.DEVICE))
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        total += loss.item()
    return total / len(loader)

@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    preds, labels = [], []
    for batch in loader:
        logits = model(
            batch['input_ids'].to(Config.DEVICE),
            batch['attention_mask'].to(Config.DEVICE),
            batch['image'].to(Config.DEVICE)
        )
        preds.extend(logits.argmax(1).cpu().numpy())
        labels.extend(batch['label'].cpu().numpy())
    return np.array(labels), np.array(preds)

# -------------------- Main --------------------
print("Device:", Config.DEVICE)

tokenizer = AutoTokenizer.from_pretrained(Config.TEXT_MODEL)
df = pd.read_csv(Config.TRAIN_CSV)

y_all = df['Category'].astype(str).str.strip().str.lower().map(lambda x: Config.LABEL_MAP.get(x, 1))
train_df, val_df = train_test_split(df, test_size=0.15, random_state=Config.SEED, stratify=y_all)
train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)

print(f"Train samples: {len(train_df)} | Validation samples: {len(val_df)}")

train_ds = MemeDataset(train_df, Config.TRAIN_IMG, tokenizer, train_tf)
val_ds   = MemeDataset(val_df, Config.TRAIN_IMG, tokenizer, val_tf)

# Class weights
y = train_df[train_ds.label_col].astype(str).str.strip().str.lower().map(lambda x: Config.LABEL_MAP.get(x, 1)).values
class_w = torch.tensor(compute_class_weight('balanced', classes=np.arange(4), y=y),
                       dtype=torch.float).to(Config.DEVICE)
print("Class weights:", class_w.cpu().numpy().round(3))

train_loader = DataLoader(train_ds, batch_size=Config.BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds, batch_size=Config.BATCH_SIZE*2, shuffle=False, num_workers=2, pin_memory=True)

model = MultimodalClassifier().to(Config.DEVICE)
optimizer = torch.optim.AdamW(model.parameters(), lr=Config.LR, weight_decay=0.01)
criterion = nn.CrossEntropyLoss(weight=class_w)

best_macro = 0
best_state = None
patience = 4
no_improve = 0

for epoch in range(1, Config.EPOCHS + 1):
    loss = train_epoch(model, train_loader, criterion, optimizer)
    true, pred = evaluate(model, val_loader)

    acc = accuracy_score(true, pred)
    macro = f1_score(true, pred, average='macro', zero_division=0)

    print(f"\nEpoch {epoch:02d} | Loss {loss:.4f} | Acc {acc:.4f} | Macro-F1 {macro:.4f}")
    print(classification_report(true, pred, target_names=list(Config.REV_LABEL.values()),
                                digits=4, zero_division=0))

    if macro > best_macro:
        best_macro = macro
        no_improve = 0
        best_state = copy.deepcopy(model.state_dict())
        print(">>> New best model saved")
    else:
        no_improve += 1
        if no_improve >= patience:
            print("Early stopping")
            break

# Load best model
model.load_state_dict(best_state)
torch.save(best_state, os.path.join(Config.OUTPUT_DIR, "best_model.pt"))
print("\nBest Macro-F1 on validation:", round(best_macro, 4))

# -------------------- Test Predictions --------------------
test_df = pd.read_csv(Config.TEST_CSV)
test_ds = MemeDataset(test_df, Config.TEST_IMG, tokenizer, val_tf, is_test=True)
test_loader = DataLoader(test_ds, batch_size=Config.BATCH_SIZE, shuffle=False, num_workers=2)

model.eval()
ids, preds = [], []
with torch.no_grad():
    for batch in test_loader:
        logits = model(
            batch['input_ids'].to(Config.DEVICE),
            batch['attention_mask'].to(Config.DEVICE),
            batch['image'].to(Config.DEVICE)
        )
        preds.extend(logits.argmax(1).cpu().numpy())
        ids.extend(batch['id'])

sub = pd.DataFrame({"id": ids, "label": [Config.REV_LABEL[p] for p in preds]})
sub.to_csv(os.path.join(Config.OUTPUT_DIR, "predictions.csv"), index=False)

print("\n[Success] predictions.csv saved")
print(sub['label'].value_counts())
print(sub.head(10))

Device: cuda


config.json:   0%|          | 0.00/662 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/452 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

... (2 outputs left out)

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 203MB/s] 



Epoch 01 | Loss 1.4406 | Acc 0.3969 | Macro-F1 0.3130
              precision    recall  f1-score   support

     Sarcasm     0.7342    0.2915    0.4173       199
     Neutral     0.2518    0.6195    0.3581       113
   Offensive     0.0000    0.0000    0.0000         8
Motivational     0.6145    0.3893    0.4766       131

    accuracy                         0.3969       451
   macro avg     0.4001    0.3251    0.3130       451
weighted avg     0.5655    0.3969    0.4123       451

>>> New best model saved

Epoch 02 | Loss 1.1756 | Acc 0.6142 | Macro-F1 0.4953
              precision    recall  f1-score   support

     Sarcasm     0.7674    0.6633    0.7116       199
     Neutral     0.3982    0.3982    0.3982       113
   Offensive     0.5000    0.1250    0.2000         8
Motivational     0.6037    0.7557    0.6712       131

    accuracy                         0.6142       451
   macro avg     0.5673    0.4856    0.4953       451
weighted avg     0.6226    0.6142    0.6123       